# LCS2 benchmark 

This notebook clones the code repository, runs every step of the benchmark in its own cell, and packs
`results/` into `results.zip` in the last cell.

**Before you start (once):**
1. On Hugging Face, accept the licence of `meta-llama/Llama-3.2-3B-Instruct`, and create a read token.
2. In this notebook: **Settings → Accelerator: GPU T4 x2** (one GPU is used), **Internet: On**,
   **Add-ons → Secrets**: add `HF_TOKEN` and
   switch them on for this notebook.
3. Put your repository URL in the configuration cell below.

**How to run:** run cells 1–6 interactively to check the setup, then use **Save Version → Save & Run
All (Commit)**, which keeps running with the browser closed (up to 12 h). Every step's output is shown
under its cell and also saved to `results/logs/<step>.log`.

In [1]:
# --- edit these ---------------------------------------------------------------
REPO_URL = "https://github.com/aarishshahmohsin/lcs2"   # the repository you uploaded
BRANCH   = "main"
MODELS   = ["llama", "qwen"]       # run order; one model is loaded at a time
RESUME   = False                   # True = continue from a previous version's results (attached as input)
RUN_EXTRA = True                   # exploratory reasoning-first experiment (about 260 generations per model)
BATCH_SIZE = 8                     # keep 8 for the final run (batch composition affects float16 ties)
# ------------------------------------------------------------------------------
WORK = "/kaggle/working"
REPO = f"{WORK}/repo"
OUT  = f"{WORK}/results"           # results are written here and zipped at the end
print("results ->", OUT)

results -> /kaggle/working/results


## GPU Check

In [2]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator -> GPU T4"
print(torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0))
print("CUDA kernel test:", (torch.ones(2, device="cuda") * 2).tolist())

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04
torch 2.11.0+cu128 | CUDA available: True
Tesla T4 | compute capability (7, 5)
CUDA kernel test: [2.0, 2.0]


## Cloning the Repository

In [3]:
import os, subprocess, glob
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
url = REPO_URL
try:                                   # private repository: use a GitHub token from Kaggle Secrets
    gh = secrets.get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{gh}@")
except Exception:
    pass
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO], check=True)
else:
    print("repository already present:", REPO)
hits = glob.glob(f"{REPO}/**/step1_setup.py", recursive=True)
assert hits, "step1_setup.py not found in the repository"
CODE = os.path.dirname(hits[0])
commit = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code directory:", CODE)
print("commit:", commit)

Cloning into '/kaggle/working/repo'...


code directory: /kaggle/working/repo
commit: 9ba8af2e9ba5b0ac7a05588cc59ef1cf29f5eca8


## Packages

In [4]:
!pip -q install -U "transformers>=4.45" accelerate huggingface_hub tabulate
import transformers, accelerate, huggingface_hub, scipy, pandas, matplotlib
print("transformers", transformers.__version__, "| accelerate", accelerate.__version__,
      "| huggingface_hub", huggingface_hub.__version__, "| scipy", scipy.__version__, "| pandas", pandas.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 42.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
transformers 5.20.0 | accelerate 1.15.0 | huggingface_hub 1.33.0 | scipy 1.16.3 | pandas 2.3.3


## Output folder and run script

In [5]:
import shutil, sys, time, json
from datetime import datetime, timezone
if RESUME and not os.path.exists(f"{OUT}/raw"):
    prev = [p for p in glob.glob("/kaggle/input/**/results/raw", recursive=True)]
    assert prev, "RESUME=True but no previous results/ attached as input"
    shutil.copytree(os.path.dirname(prev[0]), OUT, dirs_exist_ok=True)
    print("resumed from", os.path.dirname(prev[0]))
elif not RESUME and os.path.exists(f"{OUT}/raw"):
    print("WARNING: results/ already has outputs in this session; finished items will be skipped.")
os.makedirs(f"{OUT}/logs", exist_ok=True)
with open(f"{OUT}/logs/run_info.json", "w") as f:
    json.dump(dict(repo=REPO_URL, branch=BRANCH, commit=commit, gpu=torch.cuda.get_device_name(0),
                   started_utc=datetime.now(timezone.utc).isoformat(), models=MODELS, batch_size=BATCH_SIZE), f, indent=2)

def run(script, args=(), out=True):
    """Run one step as its own process (one model loaded at a time), stream its output into this
    cell and into results/logs/<script>.log, and stop the notebook if the step fails."""
    cmd = [sys.executable, "-u", os.path.join(CODE, script)] + (["--out", OUT] if out else []) + list(args)
    log = os.path.join(OUT, "logs", script.replace(".py", ".log"))
    t0 = time.time()
    with open(log, "a", encoding="utf-8") as f:
        f.write(f"\n===== {datetime.now(timezone.utc).isoformat()} {' '.join(cmd)}\n")
        p = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            f.write(line)
        rc = p.wait()
        f.write(f"===== exit {rc} after {(time.time() - t0) / 60:.1f} min\n")
    print(f"\n[{script}] exit code {rc}, {(time.time() - t0) / 60:.1f} min")
    if rc != 0:
        raise RuntimeError(f"{script} failed (exit {rc}); see {log}. Fix it, then rerun this cell: finished items are skipped.")

GEN = ["--model", *MODELS, "--batch-size", str(BATCH_SIZE)]
print("ready")

ready


## HF login

In [6]:
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
from huggingface_hub import HfApi
for m in ["meta-llama/Llama-3.2-3B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]:
    print(m, "->", HfApi().model_info(m).sha)

meta-llama/Llama-3.2-3B-Instruct -> 0cb88a4f764b7a12671c53f0838cd831a0843b95
Qwen/Qwen2.5-3B-Instruct -> aa8e72537993ba99e69dfaafa59ed015b17504d1


## Self Test

In [7]:
run("selftest.py", out=False)

SELFTEST PASSED — synthetic outputs analysed in /tmp/lcs2_selftest_7u2f3ass
figures: ['A_baseline_reliability.png', 'B1_misleading.png', 'B2_logit_lens.png', 'B3_fairness.png', 'B4_security.png', 'C_cross.png', 'D_intervention.png', 'X_reasoning.png']

[selftest.py] exit code 0, 0.1 min


## 8. Step 1: dataset download and validation

In [8]:
run("step1_setup.py")

2026-10-16 07:30:45,942 INFO HTTP Request: GET https://huggingface.co/api/datasets/Johndfm/MedMisBench-MentalHealth/commits/main "HTTP/1.1 200 OK"
2026-10-16 07:30:45,969 INFO HTTP Request: HEAD https://huggingface.co/datasets/Johndfm/MedMisBench-MentalHealth/resolve/9757095598b6555c3e935679fdb7384aed4eb651/medmisbench_mentalhealth.jsonl "HTTP/1.1 307 Temporary Redirect"
2026-10-16 07:30:45,991 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Johndfm/MedMisBench-MentalHealth/9757095598b6555c3e935679fdb7384aed4eb651/medmisbench_mentalhealth.jsonl?%2Fdatasets%2FJohndfm%2FMedMisBench-MentalHealth%2Fresolve%2F9757095598b6555c3e935679fdb7384aed4eb651%2Fmedmisbench_mentalhealth.jsonl=&etag=%226c4c6abbb8d10e288d3dba6770ecf2f6d03aacfe%22 "HTTP/1.1 200 OK"
2026-10-16 07:30:46,029 INFO HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/Johndfm/MedMisBench-MentalHealth/9757095598b6555c3e935679fdb7384aed4eb651/medmisbench_mentalhealth.jsonl?%2Fdatasets%2FJ

### 9. Step 2: Part A baseline (Core, 200 items × 2 models)

In [9]:
run("step2_baseline.py", GEN)

2026-10-16 07:30:46,761 INFO HTTP Request: GET https://huggingface.co/api/datasets/Johndfm/MedMisBench-MentalHealth/commits/main "HTTP/1.1 200 OK"
2026-10-16 07:30:57,471 INFO NumExpr defaulting to 4 threads.
2026-10-16 07:30:58,059 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct "HTTP/1.1 200 OK"
2026-10-16 07:30:58,089 INFO HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/0cb88a4f764b7a12671c53f0838cd831a0843b95/config.json "HTTP/1.1 200 OK"
2026-10-16 07:30:58,122 INFO HTTP Request: GET https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/0cb88a4f764b7a12671c53f0838cd831a0843b95/config.json "HTTP/1.1 200 OK"
2026-10-16 07:30:58,805 INFO HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/0cb88a4f764b7a12671c53f0838cd831a0843b95/tokenizer_config.json "HTTP/1.1 200 OK"
2026-10-16 07:30:58,834 INFO HTTP Request: GET https://huggingface.co/meta-llama/Llama-3.2-3B-Instruc

### 10. Step 3: B1 misleading context (Stress, 60 × 2)

In [10]:
run("step3_hallucination.py", GEN)

2026-10-16 07:41:49,996 INFO HTTP Request: GET https://huggingface.co/api/datasets/Johndfm/MedMisBench-MentalHealth/commits/main "HTTP/1.1 200 OK"
2026-10-16 07:41:50,028 INFO llama: using the revision pinned by earlier runs: 0cb88a4f764b7a12671c53f0838cd831a0843b95
2026-10-16 07:41:56,743 INFO NumExpr defaulting to 4 threads.
2026-10-16 07:41:57,527 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/0cb88a4f764b7a12671c53f0838cd831a0843b95/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-16 07:41:57,557 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/0cb88a4f764b7a12671c53f0838cd831a0843b95?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-16 07:41:58,840 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct "HTTP/1.1 200 OK"

Loading weights: 100%|██████████| 254/254 [00:03<00:00, 72.44it/s]
2026-10-16 07:42:04,956 I

### 11. Step 4: B2 interpretability (Interp, 40 × 2)

In [11]:
run("step4_interpretability.py", ["--model", *MODELS, "--batch-size", str(BATCH_SIZE)])

2026-10-16 07:45:22,264 INFO HTTP Request: GET https://huggingface.co/api/datasets/Johndfm/MedMisBench-MentalHealth/commits/main "HTTP/1.1 200 OK"
2026-10-16 07:45:22,296 INFO llama: using the revision pinned by earlier runs: 0cb88a4f764b7a12671c53f0838cd831a0843b95
2026-10-16 07:45:28,525 INFO NumExpr defaulting to 4 threads.
2026-10-16 07:45:29,215 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/0cb88a4f764b7a12671c53f0838cd831a0843b95/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-16 07:45:29,245 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/0cb88a4f764b7a12671c53f0838cd831a0843b95?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-16 07:45:30,492 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct "HTTP/1.1 200 OK"

Loading weights: 100%|██████████| 254/254 [00:03<00:00, 74.25it/s]
2026-10-16 07:45:41,515 I

### 12. Step 5: B3 fairness (6 metadata variants × 60 × 2)

In [12]:
run("step5_fairness.py", GEN)

2026-10-16 07:52:32,456 INFO HTTP Request: GET https://huggingface.co/api/datasets/Johndfm/MedMisBench-MentalHealth/commits/main "HTTP/1.1 200 OK"
2026-10-16 07:52:32,489 INFO llama: using the revision pinned by earlier runs: 0cb88a4f764b7a12671c53f0838cd831a0843b95
2026-10-16 07:52:38,705 INFO NumExpr defaulting to 4 threads.
2026-10-16 07:52:39,380 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/0cb88a4f764b7a12671c53f0838cd831a0843b95/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-16 07:52:39,415 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/0cb88a4f764b7a12671c53f0838cd831a0843b95?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-16 07:52:40,705 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct "HTTP/1.1 200 OK"

Loading weights: 100%|██████████| 254/254 [00:03<00:00, 74.31it/s]
2026-10-16 07:52:46,691 I

### 13. Step 6: B4 prompt injection and canary (2 × 60 × 2)

In [13]:
run("step6_security.py", GEN)

2026-10-16 08:09:43,736 INFO HTTP Request: GET https://huggingface.co/api/datasets/Johndfm/MedMisBench-MentalHealth/commits/main "HTTP/1.1 200 OK"
2026-10-16 08:09:43,769 INFO llama: using the revision pinned by earlier runs: 0cb88a4f764b7a12671c53f0838cd831a0843b95
2026-10-16 08:09:50,376 INFO NumExpr defaulting to 4 threads.
2026-10-16 08:09:51,174 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/0cb88a4f764b7a12671c53f0838cd831a0843b95/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-16 08:09:51,217 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/0cb88a4f764b7a12671c53f0838cd831a0843b95?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-16 08:09:52,616 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct "HTTP/1.1 200 OK"

Loading weights: 100%|██████████| 254/254 [00:03<00:00, 73.30it/s]
2026-10-16 08:09:58,831 I

### 14. Step 7: C cross-dimension analysis (no model)

In [14]:
run("step7_cross_dimension.py", [])

2026-10-16 08:16:44,516 INFO NumExpr defaulting to 4 threads.
# LCS2 benchmark — results summary (auto-generated from saved artifacts)
Generated 2026-10-16T08:16:46.347405+00:00. Do not edit by hand; rerun the script.
- **llama**: `meta-llama/Llama-3.2-3B-Instruct` @ `0cb88a4f764b7a12671c53f0838cd831a0843b95`, dtype=float16, quant=None, transformers=5.20.0, torch=2.11.0+cu128, GPU=Tesla T4
- **qwen**: `Qwen/Qwen2.5-3B-Instruct` @ `aa8e72537993ba99e69dfaafa59ed015b17504d1`, dtype=float16, quant=None, transformers=5.20.0, torch=2.11.0+cu128, GPU=Tesla T4
- Dataset `Johndfm/MedMisBench-MentalHealth` rev `9757095`; seed 42; clean-prompt fingerprint `3016efcfb236`.

## Hypotheses (stated before analysis)

**H1 (misinformation susceptibility without uncertainty signal).** Appending the unverified misleading context lowers stress-set accuracy (McNemar chi-square p < 0.05) and, for items that flip correct→incorrect, verbal confidence does not drop by more than 5 points on average. Falsified if

### 15. Step 8: D intervention (4 hardened conditions × 60 × 2)

In [15]:
run("step8_intervention.py", GEN)

2026-10-16 08:16:48,525 INFO HTTP Request: GET https://huggingface.co/api/datasets/Johndfm/MedMisBench-MentalHealth/commits/main "HTTP/1.1 200 OK"
2026-10-16 08:16:48,558 INFO llama: using the revision pinned by earlier runs: 0cb88a4f764b7a12671c53f0838cd831a0843b95
2026-10-16 08:16:55,175 INFO NumExpr defaulting to 4 threads.
2026-10-16 08:16:55,891 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/0cb88a4f764b7a12671c53f0838cd831a0843b95/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-16 08:16:55,921 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/0cb88a4f764b7a12671c53f0838cd831a0843b95?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-16 08:16:57,320 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct "HTTP/1.1 200 OK"

Loading weights: 100%|██████████| 254/254 [00:03<00:00, 71.61it/s]
2026-10-16 08:17:03,611 I

### 16. Extra: reasoning before answering (exploratory)

In [16]:
run("extra_reasoning.py", GEN)

2026-10-16 08:32:06,819 INFO HTTP Request: GET https://huggingface.co/api/datasets/Johndfm/MedMisBench-MentalHealth/commits/main "HTTP/1.1 200 OK"
2026-10-16 08:32:06,852 INFO llama: using the revision pinned by earlier runs: 0cb88a4f764b7a12671c53f0838cd831a0843b95
2026-10-16 08:32:13,493 INFO NumExpr defaulting to 4 threads.
2026-10-16 08:32:14,203 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/0cb88a4f764b7a12671c53f0838cd831a0843b95/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-16 08:32:14,243 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/0cb88a4f764b7a12671c53f0838cd831a0843b95?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-16 08:32:15,658 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct "HTTP/1.1 200 OK"

Loading weights: 100%|██████████| 254/254 [00:03<00:00, 72.84it/s]
2026-10-16 08:32:21,928 I

### 17. Step 9: full report (`results/summary.md`)

In [17]:
run("step9_report.py", [])

2026-10-16 08:58:45,760 INFO NumExpr defaulting to 4 threads.
2026-10-16 08:58:52,900 INFO wrote /kaggle/working/results/summary.md (216 lines)
[written] /kaggle/working/results/summary.md  (tables in /kaggle/working/results/tables, case studies in /kaggle/working/results/tables/case_studies.md)

[step9_report.py] exit code 0, 0.1 min


## 18. Check the run

In [18]:
for m in MODELS:
    j = json.load(open(f"{OUT}/raw/{m}/run_manifest.json"))
    print(m, j["model_revision"], sorted({(r.get("versions") or {}).get("gpu") for r in j.get("runs", [])}))
s = open(f"{OUT}/summary.md", encoding="utf-8").read()
print(s.splitlines()[1])
start = s.index("## Brief-compliance checklist"); end = s.index("## Run inventory")
print(s[start:end])
print("failed sections:", s.count("failed** --"))

llama 0cb88a4f764b7a12671c53f0838cd831a0843b95 ['Tesla T4']
qwen aa8e72537993ba99e69dfaafa59ed015b17504d1 ['Tesla T4']
Generated 2026-10-16T08:58:49.193905+00:00. Do not edit by hand; rerun the script.
## Brief-compliance checklist

*Table `00_compliance_checklist.csv`*

| check                                                 | status   | detail                                                                                 |
|:------------------------------------------------------|:---------|:---------------------------------------------------------------------------------------|
| llama: model id                                       | OK       | meta-llama/Llama-3.2-3B-Instruct                                                       |
| llama: model revision pinned                          | OK       | 0cb88a4f764b7a12671c53f0838cd831a0843b95                                               |
| llama: one model revision across all conditions       | OK       | 0cb88a4f764b7a12671c53f0838

## 19. Download `results.zip`

In [19]:
with open(f"{OUT}/logs/run_info.json") as f:
    info = json.load(f)
info["finished_utc"] = datetime.now(timezone.utc).isoformat()
with open(f"{OUT}/logs/run_info.json", "w") as f:
    json.dump(info, f, indent=2)
os.chdir(WORK)
if os.path.exists("results.zip"):
    os.remove("results.zip")
shutil.make_archive("results", "zip", root_dir=WORK, base_dir="results")
print(f"results.zip: {os.path.getsize('results.zip') / 1e6:.1f} MB")
from IPython.display import FileLink, display
display(FileLink("results.zip"))

results.zip: 4.1 MB


/kaggle/working/results.zip